In [1]:
import lightgbm as lgb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.datasets import make_regression

In [2]:
source_file_path = 'data/co2_source.xlsx'
source_sheets = ['Coal', 'Natural gas', 'Petroleum']

sector_file_path = 'data/co2_sector.xlsx'
sector_sheets = ['Residential', 'Commercial', 'Industrial', 'Transportation', 'Electric power', 'Total']

def load_excel_data(path, sheet_list):
    merged_df = None
    for sheet in sheet_list:
        
        df = pd.read_excel(path, sheet_name=sheet, skiprows=2)
        
        
        df_melt = df.melt(id_vars=['State'], var_name='Year', value_name=sheet)
        df_melt['Year'] = pd.to_numeric(df_melt['Year'], errors='coerce')
        
        if merged_df is None:
            merged_df = df_melt
        else:
            merged_df = pd.merge(merged_df, df_melt, on=['State', 'Year'], how='outer')
            
    
    merged_df = merged_df.dropna(subset=['Year'])
    merged_df['Year'] = merged_df['Year'].astype(int)
    merged_df = merged_df[merged_df['State'] != 'US'].dropna()
    return merged_df

df_sector = load_excel_data(sector_file_path, sector_sheets)
df_source = load_excel_data(source_file_path, source_sheets)

df_clean = pd.merge(df_sector, df_source, on=['State', 'Year'], how='inner')

states = df_clean['State'].unique()

predictions = {}

df_clean['State'] = df_clean['State'].astype('category')

In [3]:
X = df_clean[['Year','State']]
y = df_clean['Total']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
}

gbm = lgb.train(params, train_data, num_boost_round=100, valid_sets=[test_data])

In [9]:
import numpy as np

y_pred = gbm.predict(X_test, num_iteration=gbm.best_iteration)
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred)))
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print("R2: ", r2)
print("MSE: ", mse)
print("MAE: ", mae)

RMSE: 10.543592631086318
R2:  0.9876403344002884
MSE:  111.16734557029771
MAE:  5.021571969938036
